# LightGBM — Fast Gradient Boosting for Large Datasets

## What is LightGBM?

LightGBM (Light Gradient Boosting Machine) is Microsoft's **blazing-fast gradient boosting library**. It achieves the same accuracy as XGBoost but is typically **5-10× faster** and uses **less memory**, making it the go-to choice for large datasets.

**Real-world analogy**: XGBoost is like a thorough student who reads every page before highlighting. LightGBM is like an experienced student who quickly skips to the important sections — it gets to the same understanding faster by being smarter about where to look.

## Why LightGBM?

| Feature | XGBoost | LightGBM |
|---------|---------|----------|
| Tree growth | Level-wise (by depth) | Leaf-wise (deepest leaf first) |
| Speed | Fast | 5-10× faster |
| Memory | High | Low (histogram-based) |
| Categorical support | Requires encoding | Native support |
| Best for | Medium datasets | Large datasets (1M+ rows) |
| Overfitting risk | Lower | Higher (need `num_leaves` control) |

## Leaf-wise vs Level-wise Growth

**Level-wise** (XGBoost): Grow all leaves at the same depth. Balanced trees.

**Leaf-wise** (LightGBM): At each step, grow the single leaf that gives the maximum loss reduction. Much faster convergence, but can overfit if `num_leaves` is too large.

**Analogy**: Level-wise is like a construction crew that finishes each floor before starting the next. Leaf-wise is like building where each day the crew works on whichever room will have the most impact — faster progress, but needs careful oversight.

## Prerequisites
- Scikit-Learn basics
- XGBoost concepts (gradient boosting)

---

**Official Docs**: https://lightgbm.readthedocs.io/  
**LightGBM Paper**: Ke et al. (2017) — https://papers.nips.cc/paper/2017/hash/6449f44a102fde848669bdd9eb6b76fa-Abstract.html  
**YouTube**: https://www.youtube.com/watch?v=n_ZMQj09S6w  
**Microsoft Blog**: https://www.microsoft.com/en-us/research/project/lightgbm/

In [ ]:
lightgbmtry:
    import lightgbm as lgb
    from lightgbm import LGBMClassifier, LGBMRegressor
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
    from sklearn.datasets import load_breast_cancer, make_classification
    from sklearn.metrics import roc_auc_score
    import warnings; warnings.filterwarnings("ignore")
    print(f"LightGBM {lgb.__version__} ready")
except ImportError:
    raise SystemExit("Run: pip install lightgbm scikit-learn pandas numpy matplotlib")


In [ ]:
# ── Sklearn API ────────────────────────────────────────────────────────────────
cancer = load_breast_cancer()
X, y = cancer.data, cancer.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, 
                                                      random_state=42, stratify=y)

lgbm_clf = LGBMClassifier(
    n_estimators=500,       # Number of boosting rounds
    learning_rate=0.05,     # Step size
    num_leaves=31,          # KEY parameter: max leaves per tree (controls complexity)
    max_depth=-1,           # -1 = no limit; control via num_leaves instead
    min_child_samples=20,   # Min samples in a leaf (regularize)
    subsample=0.8,          # Row sampling
    colsample_bytree=0.8,   # Feature sampling
    reg_alpha=0.1,          # L1
    reg_lambda=1.0,         # L2
    random_state=42,
    verbose=-1              # Suppress warnings
)

lgbm_clf.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    eval_metric='auc',
    callbacks=[lgb.early_stopping(stopping_rounds=30, verbose=False),
               lgb.log_evaluation(period=-1)]  # Silent
)

y_pred  = lgbm_clf.predict(X_test)
y_proba = lgbm_clf.predict_proba(X_test)[:, 1]

print("=== LGBMClassifier (Sklearn API) ===")
print(f"Best iteration: {lgbm_clf.best_iteration_}")
print(f"Accuracy:       {accuracy_score(y_test, y_pred):.4f}")
print(f"F1 Score:       {f1_score(y_test, y_pred):.4f}")
print(f"AUC-ROC:        {roc_auc_score(y_test, y_proba):.4f}")

---
## Key Hyperparameters: LightGBM

| Parameter | Effect | Typical Value |
|-----------|--------|---------------|
| `num_leaves` | **Most critical**: controls tree complexity | 20-300 (default 31) |
| `learning_rate` | Step size per round | 0.01-0.3 |
| `n_estimators` | Number of trees | 100-5000 (use early stopping) |
| `min_child_samples` | Min samples per leaf (regularize) | 10-100 |
| `subsample` | Row sampling per tree | 0.5-1.0 |
| `colsample_bytree` | Feature sampling | 0.5-1.0 |
| `reg_alpha` | L1 regularization | 0-1 |
| `reg_lambda` | L2 regularization | 0-10 |
| `bagging_freq` | Frequency of row sampling (enable subsample) | 1-10 |

> **LightGBM golden rule**: Control overfitting via `num_leaves`, `min_child_samples`, and `max_depth` (use together). Unlike XGBoost where `max_depth` is primary, LightGBM's `num_leaves` is the most impactful parameter.

In [ ]:
# ── Native API: lgb.Dataset + lgb.train ────────────────────────────────────────
dtrain = lgb.Dataset(X_train, label=y_train)
dval   = lgb.Dataset(X_test, label=y_test, reference=dtrain)

params = {
    'objective':  'binary',
    'metric':     'auc',
    'num_leaves': 31,
    'learning_rate': 0.05,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'verbose': -1,
    'seed': 42
}

callbacks = [
    lgb.early_stopping(stopping_rounds=30, verbose=False),
    lgb.log_evaluation(period=-1)  # Silence
]

evals_result = {}
model_native = lgb.train(
    params,
    dtrain,
    num_boost_round=500,
    valid_sets=[dtrain, dval],
    valid_names=['train', 'eval'],
    callbacks=callbacks,
    evals_result=evals_result
)

print(f"=== Native API ===")
print(f"Best iteration: {model_native.best_iteration}")
y_pred_native = (model_native.predict(X_test) > 0.5).astype(int)
print(f"Accuracy: {accuracy_score(y_test, y_pred_native):.4f}")
print(f"AUC:      {roc_auc_score(y_test, model_native.predict(X_test)):.4f}")

# Training curves
plt.figure(figsize=(9, 4))
plt.plot(evals_result['train']['auc'], label='Train AUC', color='steelblue')
plt.plot(evals_result['eval']['auc'],  label='Val AUC', color='red')
plt.axvline(model_native.best_iteration, color='green', linestyle='--',
             label=f'Best: {model_native.best_iteration}')
plt.title('LightGBM Training Curves')
plt.xlabel('Round'); plt.ylabel('AUC')
plt.legend(); plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# ── LightGBM's Native Categorical Support ──────────────────────────────────────
# This is LightGBM's main advantage over XGBoost!
# No need to one-hot encode — LightGBM splits categoricals optimally

np.random.seed(42)
n = 2000
df_cat = pd.DataFrame({
    'age':       np.random.randint(18, 70, n),
    'income':    np.random.exponential(50000, n),
    'city':      np.random.choice(['NYC', 'LA', 'Chicago', 'Houston', 'Phoenix',
                                    'Seattle', 'Miami', 'Denver'], n),
    'education': np.random.choice(['HS', 'Bachelor', 'Master', 'PhD'], n, 
                                   p=[0.30, 0.42, 0.20, 0.08]),
    'job_type':  np.random.choice(['Full-Time', 'Part-Time', 'Self-Employed', 'Unemployed'], n,
                                   p=[0.55, 0.15, 0.22, 0.08]),
})

# Target: loan approval
score = (df_cat['income']/100000 + 
         (df_cat['education'].map({'HS': 0, 'Bachelor': 1, 'Master': 2, 'PhD': 3}))/6 -
         (df_cat['job_type'] == 'Unemployed').astype(float) * 0.8 +
         np.random.randn(n) * 0.3)
y_cat = (score > score.median()).astype(int)

# Convert categoricals to 'category' dtype — LightGBM detects this automatically!
for col in ['city', 'education', 'job_type']:
    df_cat[col] = df_cat[col].astype('category')

X_cat_tr, X_cat_te, y_cat_tr, y_cat_te = train_test_split(
    df_cat, y_cat, test_size=0.2, random_state=42
)

# LightGBM sees category dtype and handles it natively
lgbm_cat = LGBMClassifier(n_estimators=200, num_leaves=31, random_state=42, verbose=-1)
lgbm_cat.fit(X_cat_tr, y_cat_tr,
              categorical_feature=['city', 'education', 'job_type'])  # Explicit list

y_pred_cat = lgbm_cat.predict(X_cat_te)
print("=== LightGBM with Native Categorical Features ===")
print(f"No one-hot encoding needed!")
print(f"Accuracy: {accuracy_score(y_cat_te, y_pred_cat):.4f}")
print(f"F1:       {f1_score(y_cat_te, y_pred_cat):.4f}")

# Feature importance
imp = pd.Series(lgbm_cat.feature_importances_, index=df_cat.columns).sort_values(ascending=True)
plt.figure(figsize=(7, 4))
imp.plot(kind='barh', color='lightgreen', edgecolor='seagreen')
plt.title('Feature Importance: LightGBM with Categoricals')
plt.tight_layout(); plt.show()

---
## Speed Comparison: LightGBM vs XGBoost vs sklearn GBM

In [ ]:
import time
from sklearn.ensemble import GradientBoostingClassifier
from xgboost import XGBClassifier

# Generate a moderately large dataset
np.random.seed(42)
X_big, y_big = make_classification(n_samples=50000, n_features=50, 
                                    n_informative=20, random_state=42)
X_big_tr, X_big_te, y_big_tr, y_big_te = train_test_split(X_big, y_big, test_size=0.2)

n_estimators = 200
results = []

for name, model in [
    ('sklearn GBM', GradientBoostingClassifier(n_estimators=n_estimators, random_state=42)),
    ('XGBoost',     XGBClassifier(n_estimators=n_estimators, random_state=42, verbosity=0)),
    ('LightGBM',    LGBMClassifier(n_estimators=n_estimators, random_state=42, verbose=-1)),
]:
    start = time.time()
    model.fit(X_big_tr, y_big_tr)
    train_time = time.time() - start
    auc = roc_auc_score(y_big_te, model.predict_proba(X_big_te)[:, 1])
    results.append({'Model': name, 'Train Time (s)': train_time, 'AUC': auc})
    print(f"{name:<15}: {train_time:.2f}s  AUC={auc:.4f}")

df_speed = pd.DataFrame(results)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
df_speed.plot(x='Model', y='Train Time (s)', kind='bar', ax=axes[0], color='coral', legend=False)
axes[0].set_title('Training Time (50K samples, 200 estimators)')
axes[0].set_ylabel('Seconds'); axes[0].tick_params(axis='x', rotation=0)
df_speed.plot(x='Model', y='AUC', kind='bar', ax=axes[1], color='steelblue', legend=False)
axes[1].set_title('AUC Score')
axes[1].set_ylim(0.85, 1.0); axes[1].tick_params(axis='x', rotation=0)
plt.tight_layout(); plt.show()
print(f"\nLightGBM speedup vs sklearn GBM: {df_speed.iloc[0]['Train Time (s)']/df_speed.iloc[2]['Train Time (s)']:.1f}×")

---
## Common Pitfalls

| Pitfall | Problem | Fix |
|---------|---------|-----|
| `num_leaves` too large | Overfitting (leaf-wise grows deep fast) | Keep `num_leaves < 2^max_depth`; start at 31 |
| Forgetting `verbose=-1` | Lots of verbose output | Set `verbose=-1` in constructor |
| Not passing categorical feature list | Treated as numeric | Explicitly pass `categorical_feature=['col1']` |
| `min_child_samples` too small | Overfitting on small datasets | Increase to 20-100 |
| Label encoding categoricals | Worse than native | Use `dtype='category'` + `categorical_feature` |

## Interview Q&A

**Q: What is leaf-wise tree growth and why is it faster?**  
A: Instead of growing all leaves at the same depth (level-wise), LightGBM always expands the leaf with the highest loss reduction. This means it finds the most informative splits first, achieving lower loss in fewer iterations — hence faster. Risk: can overfit. Control with `num_leaves` and `min_child_samples`.

**Q: How does LightGBM handle categoricals differently from XGBoost?**  
A: XGBoost requires categorical features to be pre-encoded (one-hot or ordinal). LightGBM has a built-in method that finds the optimal way to split categories by evaluating all possible groupings — this is often better than one-hot encoding (especially for high-cardinality categoricals) and much faster than XGBoost's approach.

**Q: What is GOSS (Gradient-based One-Side Sampling)?**  
A: LightGBM's key innovation. Instead of using all training samples for each tree, GOSS keeps all samples with large gradients (hard to fit examples) but only samples a fraction of small-gradient examples. This reduces computation while maintaining accuracy — the bulk of learning comes from hard examples.

## Resources
- **Official Docs**: https://lightgbm.readthedocs.io/
- **Paper** (NeurIPS 2017): https://papers.nips.cc/paper/2017/hash/6449f44a102fde848669bdd9eb6b76fa-Abstract.html
- **Tuning Guide**: https://lightgbm.readthedocs.io/en/latest/Parameters-Tuning.html
- **YouTube**: https://www.youtube.com/watch?v=n_ZMQj09S6w

---
## Mini Project: Large-Scale Customer Churn Prediction

Using LightGBM on a 10,000-row telecom customer churn dataset with mixed types.

In [ ]:
from lightgbm import LGBMClassifier, early_stopping, log_evaluation
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import classification_report, roc_auc_score

np.random.seed(42)
n = 10_000

# Simulate telecom churn data
df = pd.DataFrame({
    'tenure_months':       np.random.randint(1, 72, n),
    'monthly_charges':     np.random.normal(65, 30, n).clip(18, 120),
    'total_charges':       np.random.exponential(2000, n),
    'num_products':        np.random.randint(1, 6, n),
    'support_calls':       np.random.poisson(2, n),
    'contract_type':       np.random.choice(['Month-to-Month', 'One Year', 'Two Year'], n,
                                             p=[0.55, 0.25, 0.20]),
    'payment_method':      np.random.choice(['Electronic Check', 'Mailed Check',
                                              'Bank Transfer', 'Credit Card'], n),
    'internet_service':    np.random.choice(['DSL', 'Fiber Optic', 'None'], n, p=[0.35, 0.45, 0.20]),
    'senior_citizen':      np.random.binomial(1, 0.16, n),
    'has_partner':         np.random.binomial(1, 0.48, n),
    'has_dependents':      np.random.binomial(1, 0.30, n),
})

# Realistic churn: short-tenure + high charges + month-to-month → more churn
z = (
    -df['tenure_months'] / 36
    + df['monthly_charges'] / 100
    + (df['contract_type'] == 'Month-to-Month').astype(float) * 1.5
    + df['support_calls'] / 10
    + np.random.randn(n) * 0.5
)
y = (1 / (1 + np.exp(-z)) > 0.5).astype(int)
print(f"Churn rate: {y.mean():.1%}")

# Mark categoricals
for col in ['contract_type', 'payment_method', 'internet_service']:
    df[col] = df[col].astype('category')

X_tr, X_te, y_tr, y_te = train_test_split(df, y, test_size=0.2, random_state=42, stratify=y)

model = LGBMClassifier(
    n_estimators=1000, learning_rate=0.03, num_leaves=31,
    min_child_samples=30, subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=1.0, class_weight='balanced',
    random_state=42, verbose=-1
)
model.fit(X_tr, y_tr,
           eval_set=[(X_te, y_te)],
           categorical_feature=['contract_type', 'payment_method', 'internet_service'],
           callbacks=[early_stopping(30, verbose=False), log_evaluation(-1)])

y_pred  = model.predict(X_te)
y_proba = model.predict_proba(X_te)[:, 1]
print(f"\nAUC-ROC: {roc_auc_score(y_te, y_proba):.4f}")
print(f"Best iteration: {model.best_iteration_}")
print(classification_report(y_te, y_pred, target_names=['Retained', 'Churned']))

# Feature importance
feat_imp = pd.Series(model.feature_importances_, index=df.columns).sort_values(ascending=False)
print("\nTop features:", feat_imp.head(5).to_dict())